In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import seaborn as sns
import textwrap

# ==============================================================================
# BLOK 0: KONFIGURACJA BADANIA (SETUP)
# ==============================================================================
# Parametry modelu
HIPOTEZA = "H1: Rozwój OZE w Niemczech wykazuje trend deterministyczny, który może zostać trwale zakłócony jedynie przez silne szoki strukturalne w otoczeniu makroekonomicznym."
ZMIENNA_OBJASNIANA = 'res_share'  # Y
ZMIENNE_OBJASIAJACE = ['time_index'] # X (w modelu trendu)

# Wczytanie i czyszczenie danych
try:
    df = pd.read_csv('dane_miesieczne_DE.csv')
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    df = df.sort_values('date')
    df['time_index'] = np.arange(len(df))
except FileNotFoundError:
    print("CRITICAL ERROR: Brak pliku danych. Wgraj plik .csv!")

# ==============================================================================
# BLOK 1: MODELOWANIE EKONOMETRYCZNE (PIOTR - SZEF TECHNICZNY)
# ==============================================================================
# 1.1. Estymacja klasyczną Metodą Najmniejszych Kwadratów (MNK/OLS)
X = df[['time_index']]
y = df[ZMIENNA_OBJASNIANA]

model = LinearRegression()
model.fit(X, y)

# 1.2. Diagnostyka modelu (To różni amatora od ekonometryka)
y_pred = model.predict(X)
residuals = y - y_pred # Reszty (błędy) modelu
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

# Parametry trendu
slope = model.coef_[0]
intercept = model.intercept_

# ==============================================================================
# BLOK 2: GENEROWANIE SCENARIUSZY (SYMULACJA SZOKÓW)
# ==============================================================================
# Prognoza ex-ante (do 2030)
last_date = df['date'].iloc[-1]
last_index = df['time_index'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), end='2030-12-31', freq='M')
future_X = np.arange(last_index + 1, last_index + 1 + len(future_dates)).reshape(-1, 1)

# Scenariusz Bazowy (Ceteris Paribus - przy pozostałych warunkach niezmienionych)
forecast_base = model.predict(future_X)

# Definicja czynników zakłócających (szoki egzogeniczne)
# Tabela ekspercka - Klaudia
factors_data = {
    'Sfera': ['Technologia', 'Makroekonomia', 'Geopolityka', 'Klimat', 'Legislacja'],
    'Szok': [
        'Przełom w H2 (Wodór)',        # Pozytywny szok podażowy
        'Stagflacja w UE',             # Negatywny szok popytowy
        'Kryzys surowcowy (węgiel)',   # Szok podażowy
        'Anomalie pogodowe',           # Szok losowy
        'Deregulacja OZE'              # Pozytywny szok prawny
    ],
    'Kierunek': [1, -1, -1, 1, 1], # 1 = wzrost OZE, -1 = spadek
    'Waga_Ekspercka': [5, 4, 3, 4, 2], # Siła wpływu (1-5)
    'Prawdopodobienstwo': [0.20, 0.40, 0.30, 0.60, 0.50]
}
factors_df = pd.DataFrame(factors_data)

# Obliczenie "Impact Factor" dla lejka
# Metodologia: Wartość oczekiwana szoku = Waga * Prawdopodobieństwo
factors_df['Impact_Factor'] = factors_df['Waga_Ekspercka'] * factors_df['Prawdopodobienstwo'] * factors_df['Kierunek']

total_positive_shock = factors_df[factors_df['Impact_Factor'] > 0]['Impact_Factor'].sum()
total_negative_shock = factors_df[factors_df['Impact_Factor'] < 0]['Impact_Factor'].sum()

# Kalibracja wrażliwości modelu (Sensitivity Analysis)
# Założenie: 1 jednostka szoku zmienia nachylenie trendu o 15% (parametr ekspercki)
sensitivity_param = slope * 0.15

forecast_opt = forecast_base + (future_X.flatten() - last_index) * (total_positive_shock * sensitivity_param)
forecast_pes = forecast_base + (future_X.flatten() - last_index) * (total_negative_shock * sensitivity_param) # negative shock is already negative

# ==============================================================================
# BLOK 3: OUTPUT DLA ZESPOŁU (KARTY PRACY)
# ==============================================================================

print("="*60)
print("RAPORT DIAGNOSTYCZNY - DO ROZDZIELENIA ZADAŃ")
print("="*60)

print(f"\n[DLA PIOTRA] - Sekcja Metodologiczna i Diagnostyka")
print(f"1. Model: Y = {intercept:.4f} + {slope:.4f} * t + e")
print(f"2. Dopasowanie (R^2): {r2:.4f} (Model wyjaśnia {r2*100:.1f}% zmienności historycznej)")
print(f"3. Błąd standardowy (RMSE): {rmse:.4f}")
print("   Zadanie: Opisz, czy reszty mają rozkład normalny (patrz wykres w PDF).")
print("   Jeśli reszty układają się w wzór, model może pomijać zmienną cykliczną.")

print(f"\n[DLA IZY] - Sekcja Makroekonomiczna i Kontekst")
print("   Zadanie: Przeanalizuj tabelę korelacji w PDF.")
print("   Pytanie badawcze: Dlaczego korelacja OZE z ceną jest niska/wysoka?")
print("   Hipoteza do weryfikacji: Czy 'Merit Order Effect' (tanie OZE obniża ceny) jest widoczny w danych?")
print(f"   Dane pomocnicze: Średnia cena historyczna = {df['market_price'].mean():.2f} EUR/MWh")

print(f"\n[DLA KLAUDII] - Sekcja Strategiczna (Foresight)")
print("   Zadanie: Uzasadnij dobór wag w tabeli czynników zakłócających.")
print("   Pytanie badawcze: Który z tych czynników jest 'Czarnym Łabędziem' (małe prawd., duży wpływ)?")
print(f"   Wynik modelu: Scenariusz optymistyczny zakłada przyspieszenie o {(total_positive_shock * sensitivity_param)/slope:.1%} względem trendu.")
print("="*60)

# ==============================================================================
# BLOK 4: GENEROWANIE RAPORTU PDF (WIZUALIZACJA NAUKOWA)
# ==============================================================================
pdf_filename = 'Badanie_Scenariuszowe_OZE_2030.pdf'

def draw_header(plt, title, subtitle):
    plt.text(0.5, 1.05, title, ha='center', fontsize=14, weight='bold', transform=plt.gca().transAxes)
    plt.text(0.5, 1.0, subtitle, ha='center', fontsize=10, style='italic', transform=plt.gca().transAxes)

with PdfPages(pdf_filename) as pdf:

    # STRONA 1: DIAGNOSTYKA MODELU (Dla Piotra)
    fig = plt.figure(figsize=(11.69, 8.27))
    plt.suptitle("CZĘŚĆ 1: DIAGNOSTYKA MODELU EKONOMETRYCZNEGO", fontsize=16, weight='bold')

    # Wykres 1: Dopasowanie
    ax1 = fig.add_subplot(221)
    ax1.scatter(df['date'], df['res_share'], s=5, color='black', alpha=0.5, label='Obserwacje emp.')
    ax1.plot(df['date'], y_pred, color='blue', linewidth=2, label='Model teoretyczny')
    ax1.set_title(f"Dopasowanie modelu trendu (R2={r2:.2f})")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Wykres 2: Reszty (Residuals) - Kluczowe dla ekonometryka
    ax2 = fig.add_subplot(222)
    sns.histplot(residuals, kde=True, ax=ax2, color='purple')
    ax2.set_title("Rozkład reszt modelu (Czy błędy są losowe?)")
    ax2.set_xlabel("Wartość reszty")

    # Tekst techniczny
    ax3 = fig.add_subplot(212)
    ax3.axis('off')
    tech_text = (
        f"METODOLOGIA:\n"
        f"Zastosowano model regresji liniowej względem czasu (OLS). "
        f"Estymator kierunkowy wynosi {slope:.5f}, co oznacza średni miesięczny przyrost "
        f"udziału OZE o ok. {slope*100:.2f} p.p. (ceteris paribus).\n\n"
        f"WNIOSKI Z DIAGNOSTYKI:\n"
        f"Współczynnik R2 na poziomie {r2:.2f} wskazuje na silną deterministyczną "
        f"składową procesu. Analiza histogramu reszt pozwala ocenić, czy w danych występują "
        f"zjawiska nieliniowe nieujęte w modelu."
    )
    ax3.text(0.05, 0.8, tech_text, fontsize=12, va='top', fontfamily='monospace')

    pdf.savefig()
    plt.close()

    # STRONA 2: ANALIZA OTOCZENIA (Dla Izy)
    fig = plt.figure(figsize=(11.69, 8.27))
    plt.suptitle("CZĘŚĆ 2: ANALIZA OTOCZENIA MAKROEKONOMICZNEGO", fontsize=16, weight='bold')

    # Macierz korelacji
    ax1 = fig.add_subplot(121)
    # Wybieramy tylko istotne zmienne
    corr_vars = ['res_share', 'fossil_share', 'market_price', 'energy_consumption']
    mask = np.triu(np.ones_like(df[corr_vars].corr(), dtype=bool))
    sns.heatmap(df[corr_vars].corr(), mask=mask, annot=True, cmap='RdBu', center=0, ax=ax1, square=True, cbar_kws={"shrink": .5})
    ax1.set_title("Macierz Korelacji Pearsona")

    # Opis dla Izy
    ax2 = fig.add_subplot(122)
    ax2.axis('off')
    macro_text = (
        "ZADANIE BADAWCZE (IZA):\n"
        "Proszę o interpretację ekonomiczną macierzy korelacji.\n\n"
        "1. Ujemna korelacja OZE vs Paliwa Kopalne (-0.96) jest trywialna technicznie, "
        "ale kluczowa strategicznie - potwierdza substytucyjność.\n\n"
        "2. Korelacja OZE vs Cena Energii: Należy zweryfikować, czy jest dodatnia czy ujemna. "
        "Teoria 'Merit Order' sugeruje, że OZE obniża ceny (zerowy koszt krańcowy). "
        "Jeśli w danych jest inaczej (korelacja dodatnia), oznacza to, że inne czynniki "
        "(ceny gazu, CO2) dominowały w tym okresie.\n\n"
        "3. Popyt (Consumption): Sprawdź, czy spadek zużycia energii w ostatnich latach "
        "jest skorelowany ze wzrostem OZE (efektywność energetyczna)."
    )
    ax2.text(0.05, 0.9, '\n'.join(textwrap.wrap(macro_text, 40)), fontsize=12, va='top')

    pdf.savefig()
    plt.close()

    # STRONA 3: SCENARIUSZE I FORESIGHT (Dla Klaudii i Finalny Wykres)
    fig = plt.figure(figsize=(11.69, 8.27))
    plt.suptitle("CZĘŚĆ 3: PROGNOZA SCENARIUSZOWA (FORESIGHT)", fontsize=16, weight='bold')

    # Wykres Lejka
    ax = fig.add_subplot(111)

    # Historia
    ax.plot(df['date'], df['res_share'], color='black', alpha=0.6, label='Dane Historyczne (2015-2025)')

    # Prognozy
    dates_future_plot = pd.to_datetime(future_df['date'])
    ax.plot(dates_future_plot, forecast_base, color='blue', linestyle='--', linewidth=2, label='Scenariusz Bazowy (Status Quo)')
    ax.plot(dates_future_plot, forecast_opt, color='green', linewidth=2, label='Scenariusz Optymistyczny')
    ax.plot(dates_future_plot, forecast_pes, color='red', linewidth=2, label='Scenariusz Pesymistyczny')

    # Obszar niepewności
    ax.fill_between(dates_future_plot, forecast_pes, forecast_opt, color='gray', alpha=0.15, label='Stożek Niepewności')

    # Tabela czynników na wykresie
    table_text = "Założenia szoków (Klaudia):\n" + "\n".join([f"- {row['Szok']} (Waga: {row['Waga_Ekspercka']})" for i, row in factors_df.iterrows()])
    ax.text(0.02, 0.95, table_text, transform=ax.transAxes, fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_title("Projekcja udziału OZE w miksie energetycznym Niemiec do 2030 r.")
    ax.set_ylabel("Udział OZE")
    ax.legend(loc='lower right')
    ax.grid(True, linestyle=':', alpha=0.7)

    pdf.savefig()
    plt.close()

print(f"\nGenerowanie zakończone. Pobierz plik: {pdf_filename}")

/tmp/ipython-input-1379109778.py:53: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), end='2030-12-31', freq='M')
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


RAPORT DIAGNOSTYCZNY - DO ROZDZIELENIA ZADAŃ

[DLA PIOTRA] - Sekcja Metodologiczna i Diagnostyka
1. Model: Y = 0.1915 + 0.0010 * t + e
2. Dopasowanie (R^2): 0.6231 (Model wyjaśnia 62.3% zmienności historycznej)
3. Błąd standardowy (RMSE): 0.0302
   Zadanie: Opisz, czy reszty mają rozkład normalny (patrz wykres w PDF).
   Jeśli reszty układają się w wzór, model może pomijać zmienną cykliczną.

[DLA IZY] - Sekcja Makroekonomiczna i Kontekst
   Zadanie: Przeanalizuj tabelę korelacji w PDF.
   Pytanie badawcze: Dlaczego korelacja OZE z ceną jest niska/wysoka?
   Hipoteza do weryfikacji: Czy 'Merit Order Effect' (tanie OZE obniża ceny) jest widoczny w danych?
   Dane pomocnicze: Średnia cena historyczna = 72.54 EUR/MWh

[DLA KLAUDII] - Sekcja Strategiczna (Foresight)
   Zadanie: Uzasadnij dobór wag w tabeli czynników zakłócających.
   Pytanie badawcze: Który z tych czynników jest 'Czarnym Łabędziem' (małe prawd., duży wpływ)?
   Wynik modelu: Scenariusz optymistyczny zakłada przyspieszenie 